# Workbook 16 — SMS Customer Activation Analytics

## Customer Activation Intelligence System™

This workbook builds a customer activation intelligence layer for the Pizza House Grand Reopening SMS campaign.

The goal is to connect campaign delivery data, SimpleTexting contact exports, customer reply behavior, verified engagement signals, opt-outs, and future marketing segmentation into one structured analytics system.

Workbook 16 now uses the SimpleTexting Contacts Export as the canonical reply source. Screenshot and OCR workflows are no longer the primary architecture.

## 16.0 Business Context

Pizza House moved to a new location at 5050 Stockton Blvd and launched a Grand Reopening SMS campaign using SimpleTexting.

Earlier Pizza House workbooks focused on orders, revenue, demand timing, customer records, and Tableau reporting. Workbook 16 focuses on customer activation.

The campaign data includes structured delivery reports and SimpleTexting contact exports. The contact export contains customer phone numbers, imported address fields, opt-in status, unsubscribe indicators, and full Inbox conversation history.

The final objective is to build a reusable marketing database that identifies delivered messages, verified customer responses, opt-outs, invalid numbers, duplicate contacts, and future marketing-ready customers.

## 16.1 Customer Activation Framework

Workbook 16 follows a customer activation funnel:

```text
Customer List
    ↓
SMS Delivery
    ↓
SimpleTexting Contact Export
    ↓
Inbox Conversation Parsing
    ↓
Verified Customer Responses
    ↓
Phone-Based Customer Match
    ↓
Future Marketing Audience
```

The key business metric is **Verified Customer Responses**, not only exact `YES` replies.

Because coupon responses were delayed, some customers replied multiple times or used signals such as thumbs up, positive emojis, and HELP messages while waiting for the coupon. These are treated as verified engagement when the customer intent is clearly positive.

## 16.2 Setup — Imports and File Paths

This section defines the project folders used throughout Workbook 16.

The notebook follows the existing Pizza House repository structure and keeps raw source files, cleaned datasets, and final exports separated.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
CLEANED_DIR = DATA_DIR / "cleaned"
EXPORT_DIR = DATA_DIR / "exports"

SMS_RAW_DIR = RAW_DIR / "sms"
SC_DIR = SMS_RAW_DIR / "campaign_summaries"
DR_DIR = SMS_RAW_DIR / "delivery_reports"
CONTACTS_DIR = SMS_RAW_DIR / "contacts"
IMPORTS_DIR = SMS_RAW_DIR / "imports"

CLEANED_SMS_DIR = CLEANED_DIR / "sms"

folders = [
    RAW_DIR,
    CLEANED_DIR,
    EXPORT_DIR,
    SMS_RAW_DIR,
    SC_DIR,
    DR_DIR,
    CONTACTS_DIR,
    IMPORTS_DIR,
    CLEANED_SMS_DIR,
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("WB16 folders ready")
print("Campaign summaries:", SC_DIR)
print("Delivery reports:", DR_DIR)
print("SimpleTexting contacts:", CONTACTS_DIR)
print("Cleaned SMS outputs:", CLEANED_SMS_DIR)

WB16 folders ready
Campaign summaries: ../data/raw/sms/campaign_summaries
Delivery reports: ../data/raw/sms/delivery_reports
SimpleTexting contacts: ../data/raw/sms/contacts
Cleaned SMS outputs: ../data/cleaned/sms


## 16.3 Source Data Inventory

This section confirms the files available for Workbook 16.

The source data is grouped into three active categories:

- Campaign summary screenshots
- Delivery report CSV files
- SimpleTexting contact exports

This inventory step creates a quick audit trail before analysis begins.

In [2]:
sc_files = sorted([file for file in SC_DIR.glob("*") if not file.name.startswith(".")])
dr_files = sorted(DR_DIR.glob("*.csv"))
contact_files = sorted(CONTACTS_DIR.glob("*.csv"))
import_files = sorted(IMPORTS_DIR.glob("*"))

print("Campaign summary files:", len(sc_files))
for file in sc_files:
    print(" -", file.name)

print("\nDelivery report files:", len(dr_files))
for file in dr_files:
    print(" -", file.name)

print("\nSimpleTexting contact files:", len(contact_files))
for file in contact_files:
    print(" -", file.name)

print("\nImport files:", len(import_files))
for file in import_files:
    print(" -", file.name)

Campaign summary files: 8
 - d1_a_sc.png
 - d1_b_sc.png
 - d2 _sc.png
 - d3_2_sc.png
 - d3_sc.png
 - d4_sc.png
 - d5_sc.png
 - d6_sc.png

Delivery report files: 8
 - d1_a_dr.csv
 - d1_b_dr.csv
 - d2_dr.csv
 - d3_2_dr.csv
 - d3_dr.csv
 - d4_dr.csv
 - d5_dr.csv
 - d6_dr.csv

SimpleTexting contact files: 1
 - d1_contacts.csv

Import files: 0


## 16.4 Campaign Timeline

The campaign was launched in multiple waves because SimpleTexting daily sending limits required the customer list to be split into smaller batches.

The final campaign structure included:

- D1_A
- D1_B
- D2
- D3
- D3_2
- D4
- D5
- D6

The timeline is documented manually because it explains how the campaign evolved and why multiple campaign batches exist.

In [3]:
campaign_timeline = pd.DataFrame([
    {
        "campaign": "D1_A",
        "campaign_group": "Initial Test",
        "description": "First launch batch after Pizza House reopening message was prepared.",
        "notes": "Small batch used to begin campaign delivery.",
    },
    {
        "campaign": "D1_B",
        "campaign_group": "Initial Test",
        "description": "Second launch batch sent after D1_A.",
        "notes": "Continued initial campaign rollout.",
    },
    {
        "campaign": "D2",
        "campaign_group": "Scale Wave",
        "description": "Larger campaign wave after initial batches.",
        "notes": "Daily limits and platform behavior required campaign management.",
    },
    {
        "campaign": "D3",
        "campaign_group": "Restart Wave",
        "description": "Campaign wave affected by campaign restart / overlap behavior.",
        "notes": "Tracked separately to preserve accurate source attribution.",
    },
    {
        "campaign": "D3_2",
        "campaign_group": "System Constraint Wave",
        "description": "Additional D3-related campaign batch created because of platform constraints.",
        "notes": "Kept as its own campaign to avoid mixing source files.",
    },
    {
        "campaign": "D4",
        "campaign_group": "Weekend Wave",
        "description": "Follow-up campaign batch after earlier waves.",
        "notes": "Part of continued customer activation rollout.",
    },
    {
        "campaign": "D5",
        "campaign_group": "Final Wave",
        "description": "Final large customer activation wave.",
        "notes": "Sent after campaign structure was stabilized.",
    },
    {
        "campaign": "D6",
        "campaign_group": "Final Wave",
        "description": "Final remaining customer activation wave.",
        "notes": "Completed remaining customer outreach.",
    },
])

campaign_timeline.to_csv(CLEANED_SMS_DIR / "campaign_timeline.csv", index=False)
print("Exported:", CLEANED_SMS_DIR / "campaign_timeline.csv")
campaign_timeline

Exported: ../data/cleaned/sms/campaign_timeline.csv


,campaign,campaign_group,description,notes
0,D1_A,Initial Test,First launch batch after Pizza House reopening...,Small batch used to begin campaign delivery.
1,D1_B,Initial Test,Second launch batch sent after D1_A.,Continued initial campaign rollout.
2,D2,Scale Wave,Larger campaign wave after initial batches.,Daily limits and platform behavior required ca...
3,D3,Restart Wave,Campaign wave affected by campaign restart / o...,Tracked separately to preserve accurate source...
4,D3_2,System Constraint Wave,Additional D3-related campaign batch created b...,Kept as its own campaign to avoid mixing sourc...
5,D4,Weekend Wave,Follow-up campaign batch after earlier waves.,Part of continued customer activation rollout.
6,D5,Final Wave,Final large customer activation wave.,Sent after campaign structure was stabilized.
7,D6,Final Wave,Final remaining customer activation wave.,Completed remaining customer outreach.


## 16.5 Campaign Summary Table

SimpleTexting campaign summary screenshots provide campaign-level information that is not always available through delivery report exports.

This section creates a structured campaign summary table that can be updated from screenshots.

The table is intentionally manual because screenshot metrics must be verified before they are used in executive reporting.

In [4]:
campaign_summary = pd.DataFrame([
    {"campaign": "D1_A", "campaign_group": "Initial Test", "contacts": 498, "send_date": "2026-06-08", "send_time": "15:30", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D1_B", "campaign_group": "Initial Test", "contacts": 493, "send_date": "2026-06-08", "send_time": "15:45", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D2", "campaign_group": "Scale Wave", "contacts": 2468, "send_date": "2026-06-09", "send_time": "various", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D3", "campaign_group": "Restart Wave", "contacts": 996, "send_date": "2026-06-12", "send_time": "15:30", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D3_2", "campaign_group": "System Constraint Wave", "contacts": np.nan, "send_date": "2026-06-10", "send_time": "various", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update contact and delivery metrics from screenshots."},
    {"campaign": "D4", "campaign_group": "Weekend Wave", "contacts": 1501, "send_date": "2026-06-14", "send_time": "14:00", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D5", "campaign_group": "Final Wave", "contacts": 1509, "send_date": "2026-06-16", "send_time": "16:00", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
    {"campaign": "D6", "campaign_group": "Final Wave", "contacts": 1511, "send_date": "2026-06-17", "send_time": "16:00", "total_sent": np.nan, "delivered": np.nan, "failed": np.nan, "success_rate": np.nan, "notes": "Update delivery metrics from SimpleTexting campaign summary screenshot."},
])

campaign_summary["send_date"] = pd.to_datetime(campaign_summary["send_date"])

campaign_summary.to_csv(CLEANED_SMS_DIR / "campaign_summary.csv", index=False)
campaign_summary.to_csv(EXPORT_DIR / "campaign_summary.csv", index=False)

print("Exported:", CLEANED_SMS_DIR / "campaign_summary.csv")
print("Exported:", EXPORT_DIR / "campaign_summary.csv")
campaign_summary

Exported: ../data/cleaned/sms/campaign_summary.csv
Exported: ../data/exports/campaign_summary.csv


,campaign,campaign_group,contacts,send_date,send_time,total_sent,delivered,failed,success_rate,notes
0,D1_A,Initial Test,498.0,2026-06-08,15:30,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
1,D1_B,Initial Test,493.0,2026-06-08,15:45,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
2,D2,Scale Wave,2468.0,2026-06-09,various,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
3,D3,Restart Wave,996.0,2026-06-12,15:30,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
4,D3_2,System Constraint Wave,NaN,2026-06-10,various,NaN,NaN,NaN,NaN,Update contact and delivery metrics from scree...
5,D4,Weekend Wave,1501.0,2026-06-14,14:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
6,D5,Final Wave,1509.0,2026-06-16,16:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...
7,D6,Final Wave,1511.0,2026-06-17,16:00,NaN,NaN,NaN,NaN,Update delivery metrics from SimpleTexting cam...


## 16.6 Delivery Report Pipeline

Delivery reports are the structured SMS delivery source data exported from SimpleTexting.

This section imports each delivery report, standardizes column names, attaches the campaign name from the file name, and appends all reports into one delivery master table.

In [5]:
delivery_frames = []

for path in sorted(DR_DIR.glob("*_dr.csv")):
    campaign = path.stem.replace("_dr", "").upper()
    temp = pd.read_csv(path)

    temp.columns = (
        temp.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    temp["campaign"] = campaign
    temp["source_file"] = path.name
    delivery_frames.append(temp)

    print(f"Loaded {campaign}: {len(temp):,} rows")

if delivery_frames:
    delivery_master = pd.concat(delivery_frames, ignore_index=True)
else:
    delivery_master = pd.DataFrame()
    print("No delivery report files found.")

print("Delivery master records:", len(delivery_master))
delivery_master.head()

Loaded D1_A: 497 rows
Loaded D1_B: 493 rows
Loaded D2: 2,975 rows
Loaded D3_2: 1,930 rows
Loaded D3: 866 rows
Loaded D4: 1,419 rows
Loaded D5: 1,391 rows
Loaded D6: 1,348 rows
Delivery master records: 10919


,phone_number,first_name,last_name,delivery_status,campaign,source_file
0,5304094821,22242,4220 Stockton Blvd,Opted Out,D1_A,d1_a_dr.csv
1,9164902587,12882,6 Lopis Crt,Delivered,D1_A,d1_a_dr.csv
2,9168852020,7970,5622 Florin Rd #70,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv
3,9164902538,756,3610 25 Ave,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv
4,2065811608,14226,4100 49th Ave Building 2 #15,Delivered,D1_A,d1_a_dr.csv


## 16.7 Delivery Data Cleaning

This section standardizes phone numbers and delivery statuses.

The cleaned fields make it possible to summarize delivery performance, identify invalid records, find duplicate contacts, and later compare delivery records against SimpleTexting contact export records.

In [6]:
if not delivery_master.empty:
    phone_candidates = [col for col in delivery_master.columns if "phone" in col]
    phone_col = phone_candidates[0] if phone_candidates else None

    if phone_col:
        delivery_master["phone_clean"] = (
            delivery_master[phone_col]
            .astype(str)
            .str.replace(r"\D", "", regex=True)
        )
    else:
        delivery_master["phone_clean"] = np.nan

    status_candidates = [col for col in delivery_master.columns if "status" in col]
    status_col = status_candidates[0] if status_candidates else None

    if status_col:
        delivery_master["delivery_status_clean"] = (
            delivery_master[status_col]
            .astype(str)
            .str.strip()
            .str.lower()
        )
    else:
        delivery_master["delivery_status_clean"] = np.nan

    delivery_master.to_csv(CLEANED_SMS_DIR / "delivery_report_master.csv", index=False)

    print("Exported:", CLEANED_SMS_DIR / "delivery_report_master.csv")
    print("Phone column used:", phone_col)
    print("Status column used:", status_col)
else:
    print("Delivery master is empty. Add delivery report CSV files before running this section.")

delivery_master.head()

Exported: ../data/cleaned/sms/delivery_report_master.csv
Phone column used: phone_number
Status column used: delivery_status


,phone_number,first_name,last_name,delivery_status,campaign,source_file,phone_clean,delivery_status_clean
0,5304094821,22242,4220 Stockton Blvd,Opted Out,D1_A,d1_a_dr.csv,5304094821,opted out
1,9164902587,12882,6 Lopis Crt,Delivered,D1_A,d1_a_dr.csv,9164902587,delivered
2,9168852020,7970,5622 Florin Rd #70,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv,9168852020,undelivered - hard bounce
3,9164902538,756,3610 25 Ave,Undelivered - Hard Bounce,D1_A,d1_a_dr.csv,9164902538,undelivered - hard bounce
4,2065811608,14226,4100 49th Ave Building 2 #15,Delivered,D1_A,d1_a_dr.csv,2065811608,delivered


## 16.8 Delivery Status Summary

Delivery status provides the first measurement of SMS campaign execution.

This section summarizes delivery outcomes by campaign so each batch can be evaluated before customer replies are analyzed.

In [7]:
if not delivery_master.empty:
    delivery_status_summary = (
        delivery_master
        .groupby(["campaign", "delivery_status_clean"], dropna=False)
        .size()
        .reset_index(name="record_count")
        .sort_values(["campaign", "record_count"], ascending=[True, False])
    )

    delivery_status_summary.to_csv(
        CLEANED_SMS_DIR / "delivery_status_summary.csv",
        index=False
    )

    print("Exported:", CLEANED_SMS_DIR / "delivery_status_summary.csv")
else:
    delivery_status_summary = pd.DataFrame(
        columns=["campaign", "delivery_status_clean", "record_count"]
    )
    print("Delivery status summary not created because delivery master is empty.")

delivery_status_summary

Exported: ../data/cleaned/sms/delivery_status_summary.csv


,campaign,delivery_status_clean,record_count
0,D1_A,delivered,323
2,D1_A,undelivered - hard bounce,92
3,D1_A,undelivered - soft bounce,43
1,D1_A,opted out,39
4,D1_B,delivered,342
6,D1_B,undelivered - hard bounce,95
5,D1_B,opted out,29
7,D1_B,undelivered - soft bounce,27
8,D2,delivered,2005
10,D2,undelivered - hard bounce,546


## 16.9 Data Quality Outputs

Delivery reports can be used to identify invalid numbers and duplicate phone records.

These outputs are operationally valuable because they improve the quality of future marketing campaigns and reduce wasted sends.

In [8]:
if not delivery_master.empty:
    invalid_keywords = ["invalid", "failed", "undelivered", "error"]

    invalid_numbers = delivery_master[
        delivery_master["delivery_status_clean"]
        .astype(str)
        .str.contains("|".join(invalid_keywords), na=False)
    ].copy()

    duplicate_phones = (
        delivery_master[delivery_master["phone_clean"].notna()]
        .groupby("phone_clean")
        .size()
        .reset_index(name="record_count")
        .query("record_count > 1")
        .sort_values("record_count", ascending=False)
    )

    invalid_numbers.to_csv(CLEANED_SMS_DIR / "invalid_numbers.csv", index=False)
    duplicate_phones.to_csv(CLEANED_SMS_DIR / "duplicate_phones.csv", index=False)

    print("Exported:", CLEANED_SMS_DIR / "invalid_numbers.csv")
    print("Exported:", CLEANED_SMS_DIR / "duplicate_phones.csv")
    print("Invalid number records:", len(invalid_numbers))
    print("Duplicate phone records:", len(duplicate_phones))
else:
    invalid_numbers = pd.DataFrame()
    duplicate_phones = pd.DataFrame()
    print("Data quality outputs not created because delivery master is empty.")

Exported: ../data/cleaned/sms/invalid_numbers.csv
Exported: ../data/cleaned/sms/duplicate_phones.csv
Invalid number records: 2561
Duplicate phone records: 1524


## 16.10 Initial Activation Funnel

This section creates the first version of the SMS activation funnel using available delivery data.

Verified response metrics are added after the SimpleTexting contact export is parsed and classified.

In [9]:
if not delivery_master.empty:
    total_delivery_records = len(delivery_master)
    unique_phone_records = delivery_master["phone_clean"].nunique()
    invalid_record_count = len(invalid_numbers)

    activation_funnel = pd.DataFrame([
        {"stage": "Delivery Report Records", "count": total_delivery_records},
        {"stage": "Unique Phone Numbers", "count": unique_phone_records},
        {"stage": "Invalid / Failed Records", "count": invalid_record_count},
    ])
else:
    activation_funnel = pd.DataFrame([
        {"stage": "Delivery Report Records", "count": 0},
        {"stage": "Unique Phone Numbers", "count": 0},
        {"stage": "Invalid / Failed Records", "count": 0},
    ])

activation_funnel.to_csv(
    EXPORT_DIR / "sms_activation_funnel_initial.csv",
    index=False
)

print("Exported:", EXPORT_DIR / "sms_activation_funnel_initial.csv")
activation_funnel

Exported: ../data/exports/sms_activation_funnel_initial.csv


,stage,count
0,Delivery Report Records,10919
1,Unique Phone Numbers,9377
2,Invalid / Failed Records,2561


## 16.11 Import SimpleTexting Contacts

SimpleTexting contact exports provide the primary data source for customer reply analysis.

Each contact record includes the customer's phone number, imported contact information, opt-in status, unsubscribe status, and complete SMS conversation history.

These contact exports serve as the foundation for reply classification, customer matching, and SMS engagement analytics throughout the remainder of this workbook.

In [10]:
contacts_raw = pd.read_csv(
    CONTACTS_DIR / "d1_contacts.csv"
)

print(f"Contacts imported: {len(contacts_raw):,}")
display(contacts_raw.head())

Contacts imported: 498


,Number,First Name,Last Name,Email,Birthday,Note,Create Date,Opt-in method,Unsubscribed,Text,Inbox
0,2065811608,14226,4100 49th Ave Building 2 #15,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...
1,2092088773,16600,2050 53rd Ave,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...
2,2092848111,1666,5804 63 St,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...
3,2093902376,4194,2487 67 Ave,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...
4,2095619339,7959,4100 49 Ave #62,NaN,NaN,NaN,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),NaN,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...


## 16.12 Clean SimpleTexting Contacts

The SimpleTexting contacts export contains customer phone numbers, imported address fields, opt-in metadata, unsubscribe status, outbound campaign text, and full conversation history.

This section standardizes the export into a clean contacts table by normalizing column names, cleaning phone numbers, rebuilding the imported address field, and preserving the Inbox thread for reply parsing.

In [11]:
contacts_clean = contacts_raw.copy()

contacts_clean.columns = (
    contacts_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

contacts_clean["phone_clean"] = (
    contacts_clean["number"]
    .astype(str)
    .str.replace(r"\D", "", regex=True)
)

contacts_clean["address_raw"] = (
    contacts_clean["first_name"].fillna("").astype(str).str.strip()
    + " "
    + contacts_clean["last_name"].fillna("").astype(str).str.strip()
).str.strip()

contacts_clean["inbox"] = (
    contacts_clean["inbox"]
    .fillna("")
    .astype(str)
    .str.strip()
)

contacts_clean["has_inbox_thread"] = contacts_clean["inbox"].ne("")

contacts_clean["unsubscribed_flag"] = (
    contacts_clean["unsubscribed"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

contacts_clean = contacts_clean[
    [
        "phone_clean",
        "address_raw",
        "create_date",
        "opt_in_method",
        "unsubscribed_flag",
        "text",
        "inbox",
        "has_inbox_thread",
    ]
].copy()

contacts_clean.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_clean.csv",
    index=False
)

print(f"Contacts cleaned: {len(contacts_clean):,}")
print(f"Contacts with inbox thread: {contacts_clean['has_inbox_thread'].sum():,}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_clean.csv")

display(contacts_clean.head(20))

Contacts cleaned: 498
Contacts with inbox thread: 497
Exported: ../data/cleaned/sms/simpletexting_contacts_clean.csv


,phone_clean,address_raw,create_date,opt_in_method,unsubscribed_flag,text,inbox,has_inbox_thread
0,2065811608,14226 4100 49th Ave Building 2 #15,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,True
1,2092088773,16600 2050 53rd Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
2,2092848111,1666 5804 63 St,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
3,2093902376,4194 2487 67 Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
4,2095619339,7959 4100 49 Ave #62,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:37 PM > Pizza House has moved!No...,True
5,2096632174,19948 6439 Rancho Adobe Dr,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,True
6,2096651647,7452 8200 Elder Creek Rd,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:33 PM > Pizza House has moved!No...,True
7,2097522780,23916 5825 61st,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:32 PM > Pizza House has moved!No...,True
8,2098098402,20875 6060 40 Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:38 PM > Pizza House has moved!No...,True
9,2164509592,14536 3644 18 Ave,06/08/2026 10:11 AM,Import (campaign_d1_a.csv),False,NaN,06/08/2026 03:31 PM > Pizza House has moved!No...,True


## 16.13 Parse Inbox Conversations

The Inbox field contains the complete SMS conversation history for each contact.

This section separates outbound Pizza House messages from inbound customer replies by identifying the message direction contained within each conversation thread. Parsed conversation fields are used throughout the remainder of the workbook for customer engagement analysis.

In [12]:
thread_pattern = (
    r"(\d{2}/\d{2}/\d{4}\s+\d{2}:\d{2}\s+[AP]M\s+[><])"
)

parsed_threads = []

for _, row in contacts_clean.iterrows():
    outbound_messages = []
    inbound_replies = []
    inbox = str(row["inbox"])

    if inbox.strip():
        thread = re.split(thread_pattern, inbox)

        for i in range(1, len(thread), 2):
            direction = thread[i]
            message = thread[i + 1].strip()

            if direction.endswith(">"):
                outbound_messages.append(message)
            elif direction.endswith("<"):
                inbound_replies.append(message)

    parsed_threads.append({
        "phone_clean": row["phone_clean"],
        "address_raw": row["address_raw"],
        "outbound_message_count": len(outbound_messages),
        "customer_reply_count": len(inbound_replies),
        "customer_replied_flag": len(inbound_replies) > 0,
        "first_customer_reply": inbound_replies[0] if inbound_replies else "",
        "all_customer_replies": " | ".join(inbound_replies),
        "all_outbound_messages": " | ".join(outbound_messages),
    })

contacts_threads = pd.DataFrame(parsed_threads)

contacts_replies = contacts_clean.merge(
    contacts_threads,
    on=["phone_clean", "address_raw"],
    how="left"
)

contacts_replies.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_replies.csv",
    index=False
)

print(f"Contacts parsed: {len(contacts_replies):,}")
print(f"Customer replies: {contacts_replies['customer_replied_flag'].sum():,}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_replies.csv")

display(
    contacts_replies[
        [
            "phone_clean",
            "address_raw",
            "customer_reply_count",
            "first_customer_reply",
            "all_customer_replies",
        ]
    ].head(20)
)

Contacts parsed: 498
Customer replies: 65
Exported: ../data/cleaned/sms/simpletexting_contacts_replies.csv


,phone_clean,address_raw,customer_reply_count,first_customer_reply,all_customer_replies
0,2065811608,14226 4100 49th Ave Building 2 #15,0,,
1,2092088773,16600 2050 53rd Ave,0,,
2,2092848111,1666 5804 63 St,0,,
3,2093902376,4194 2487 67 Ave,0,,
4,2095619339,7959 4100 49 Ave #62,0,,
5,2096632174,19948 6439 Rancho Adobe Dr,0,,
6,2096651647,7452 8200 Elder Creek Rd,0,,
7,2097522780,23916 5825 61st,0,,
8,2098098402,20875 6060 40 Ave,0,,
9,2164509592,14536 3644 18 Ave,0,,


## 16.14 Reply Classification

Customer replies are classified using the Workbook 16 verified response framework.

Because some coupon responses were delayed, positive replies, HELP messages, short confirmation signals, and positive customer engagement are treated as verified engagement when the customer intent is clearly positive.

Coupon fulfillment is identified from outbound messages that include the Pizza House coupon code.

In [13]:
verified_terms = [
    "yes",
    "y",
    "yep",
    "yeah",
    "ok",
    "okay",
    "si",
    "sí",
    "p",
    "👍",
    "❤️",
    "😊",
    "😁",
]

question_terms = [
    "where",
    "address",
    "location",
    "cross",
    "hours",
    "open",
    "when",
    "what time",
]

wrong_number_terms = [
    "wrong",
    "not me",
    "remove",
]

negative_terms = [
    "no",
    "nah",
    "nope",
    "don't",
    "dont",
]

def classify_customer_reply(reply_text):
    value = str(reply_text).strip().lower()

    if value == "":
        return "NO_REPLY"
    if "stop" in value:
        return "STOP"
    if any(term in value for term in wrong_number_terms):
        return "WRONG_NUMBER"
    if "help" in value:
        return "VERIFIED_RESPONSE"
    if (
        value in verified_terms
        or value.startswith("yes")
        or any(term in value for term in verified_terms)
    ):
        return "VERIFIED_RESPONSE"
    if any(term in value for term in question_terms):
        return "QUESTION"
    if any(term in value for term in negative_terms):
        return "NEGATIVE"
    return "OTHER"

contacts_replies["reply_category"] = (
    contacts_replies["all_customer_replies"]
    .apply(classify_customer_reply)
)

contacts_replies["verified_response_flag"] = (
    contacts_replies["reply_category"]
    .eq("VERIFIED_RESPONSE")
)

contacts_replies["coupon_sent_flag"] = (
    contacts_replies["all_outbound_messages"]
    .str.lower()
    .str.contains(
        "thanks for supporting pizza house|pizzahouse10|10% off code",
        na=False,
        regex=True
    )
)

contacts_replies["follow_up_needed"] = (
    contacts_replies["customer_replied_flag"]
    & ~contacts_replies["coupon_sent_flag"]
    & contacts_replies["reply_category"].isin(["VERIFIED_RESPONSE", "QUESTION"])
)

reply_classification_summary = (
    contacts_replies
    .groupby("reply_category")
    .size()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

contacts_replies.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_replies_classified.csv",
    index=False
)

print(f"Contacts classified: {len(contacts_replies):,}")
print(f"Verified responses: {contacts_replies['verified_response_flag'].sum():,}")
print(f"Coupons sent: {contacts_replies['coupon_sent_flag'].sum():,}")
print(f"Follow-up needed: {contacts_replies['follow_up_needed'].sum():,}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_replies_classified.csv")

display(reply_classification_summary)

Contacts classified: 498
Verified responses: 26
Coupons sent: 26
Follow-up needed: 0
Exported: ../data/cleaned/sms/simpletexting_contacts_replies_classified.csv


,reply_category,record_count
0,NO_REPLY,433
1,STOP,39
2,VERIFIED_RESPONSE,26


## 16.15 Parse Contact Addresses

The SimpleTexting contacts export stores the imported customer address in the contact name fields.

The first numeric value often represents a SimpleTexting contact identifier rather than the customer street number. This section removes that identifier, standardizes customer addresses, and creates an `address_join_key` used for data quality validation after phone-based customer matching.

Phone number remains the canonical integration key. Address is used as a secondary QA field.

In [14]:
contacts_address = contacts_replies.copy()

parsed_columns = [
    "address_clean",
    "street_number",
    "street_name",
    "street_suffix",
    "unit",
    "address_join_key",
]

contacts_address = contacts_address.drop(
    columns=[c for c in parsed_columns if c in contacts_address.columns],
    errors="ignore"
)

STREET_SUFFIXES = {
    "ST", "AVE", "DR", "RD",
    "BLVD", "CT", "CIR",
    "WAY", "LN", "PL",
    "PKWY", "TER", "TRL", "HWY",
}

UNIT_PREFIXES = (
    "APT",
    "UNIT",
    "#",
    "ROOM",
    "RM",
    "STE",
    "SUITE",
    "BUILDING",
)

ocr_corrections = {
    "BIV": "BLVD",
    "STOCKTEN": "STOCKTON",
    "PK": "PKWY",
    "OR": "DR",
    "AVENUE": "AVE",
    "STREET": "ST",
    "ROAD": "RD",
    "DRIVE": "DR",
    "BOULEVARD": "BLVD",
    "COURT": "CT",
    "CIRCLE": "CIR",
    "PARKWAY": "PKWY",
    "LANE": "LN",
    "PLACE": "PL",
    "TERRACE": "TER",
    "TRAIL": "TRL",
    "HIGHWAY": "HWY",
}

ordinal_streets = {
    "10": "10TH",
    "18": "18TH",
    "23": "23RD",
    "25": "25TH",
    "33": "33RD",
    "35": "35TH",
    "37": "37TH",
    "39": "39TH",
    "40": "40TH",
    "42": "42ND",
    "46": "46TH",
    "47": "47TH",
    "48": "48TH",
    "49": "49TH",
    "50": "50TH",
    "51": "51ST",
    "54": "54TH",
    "55": "55TH",
    "56": "56TH",
    "61": "61ST",
    "63": "63RD",
    "66": "66TH",
    "67": "67TH",
    "69": "69TH",
}

parsed_rows = []

for address in contacts_address["address_raw"]:
    address_clean = None
    street_number = None
    street_name = None
    street_suffix = None
    unit = None
    address_join_key = None

    if pd.notna(address):
        text = str(address).upper().strip()
        text = re.sub(r"\b\d+D\d*\b", "", text)
        text = re.sub(r"@\d+", "", text)
        text = text.replace("&", " ")
        text = re.sub(r"\s+", " ", text).strip()

        for bad, good in ocr_corrections.items():
            text = re.sub(rf"\b{bad}\b", good, text)

        text = re.sub(
            r"(ST|AVE|DR|RD|BLVD|CT|CIR|WAY|LN|PL|PKWY)(APT|UNIT|ROOM|RM|STE|SUITE|BUILDING)",
            r"\1 \2",
            text
        )

        tokens = text.split()

        if (
            len(tokens) >= 3
            and tokens[0].isdigit()
            and tokens[1].isdigit()
            and tokens[2] not in STREET_SUFFIXES
        ):
            tokens = tokens[1:]

        address_clean = " ".join(tokens)

        if tokens and tokens[0].isdigit():
            street_number = tokens.pop(0)

        for i, token in enumerate(tokens):
            token_upper = token.upper()
            if (
                token_upper.startswith(UNIT_PREFIXES)
                or "#" in token_upper
            ):
                unit = " ".join(tokens[i:])
                tokens = tokens[:i]
                break

        for i, token in enumerate(tokens):
            token_upper = token.upper()
            if token_upper in STREET_SUFFIXES:
                street_suffix = token_upper
                tokens = tokens[:i]
                break

        street_name = " ".join(tokens)
        street_name = street_name.replace("&", "").strip()
        street_name = re.sub(r"\s+", " ", street_name).strip()

        if street_name in ordinal_streets:
            street_name = ordinal_streets[street_name]

        join_parts = [street_number]

        if street_name:
            join_parts.append(street_name.replace(" ", "_"))

        if street_suffix:
            join_parts.append(street_suffix)

        address_join_key = "_".join([part for part in join_parts if part])

    parsed_rows.append({
        "address_clean": address_clean,
        "street_number": street_number,
        "street_name": street_name,
        "street_suffix": street_suffix,
        "unit": unit,
        "address_join_key": address_join_key,
    })

parsed_rows = pd.DataFrame(parsed_rows)

contacts_address = pd.concat(
    [contacts_address, parsed_rows],
    axis=1
)

contacts_address.to_csv(
    CLEANED_SMS_DIR / "simpletexting_contacts_address_parsed.csv",
    index=False
)

parsed_addresses = contacts_address["address_join_key"].notna().sum()
total_contacts = len(contacts_address)
parse_rate = parsed_addresses / total_contacts

print(f"Contacts parsed: {total_contacts:,}")
print(f"Addresses parsed: {parsed_addresses:,}")
print(f"Parse rate: {parse_rate:.1%}")
print("Exported:", CLEANED_SMS_DIR / "simpletexting_contacts_address_parsed.csv")

display(
    contacts_address[
        [
            "phone_clean",
            "address_raw",
            "address_clean",
            "street_number",
            "street_name",
            "street_suffix",
            "unit",
            "address_join_key",
        ]
    ].head(20)
)

Contacts parsed: 498
Addresses parsed: 498
Parse rate: 100.0%
Exported: ../data/cleaned/sms/simpletexting_contacts_address_parsed.csv


,phone_clean,address_raw,address_clean,street_number,street_name,street_suffix,unit,address_join_key
0,2065811608,14226 4100 49th Ave Building 2 #15,4100 49TH AVE BUILDING 2 #15,4100,49TH,AVE,BUILDING 2 #15,4100_49TH_AVE
1,2092088773,16600 2050 53rd Ave,2050 53RD AVE,2050,53RD,AVE,None,2050_53RD_AVE
2,2092848111,1666 5804 63 St,5804 63 ST,5804,63RD,ST,None,5804_63RD_ST
3,2093902376,4194 2487 67 Ave,2487 67 AVE,2487,67TH,AVE,None,2487_67TH_AVE
4,2095619339,7959 4100 49 Ave #62,4100 49 AVE #62,4100,49TH,AVE,#62,4100_49TH_AVE
5,2096632174,19948 6439 Rancho Adobe Dr,6439 RANCHO ADOBE DR,6439,RANCHO ADOBE,DR,None,6439_RANCHO_ADOBE_DR
6,2096651647,7452 8200 Elder Creek Rd,8200 ELDER CREEK RD,8200,ELDER CREEK,RD,None,8200_ELDER_CREEK_RD
7,2097522780,23916 5825 61st,5825 61ST,5825,61ST,None,None,5825_61ST
8,2098098402,20875 6060 40 Ave,6060 40 AVE,6060,40TH,AVE,None,6060_40TH_AVE
9,2164509592,14536 3644 18 Ave,3644 18 AVE,3644,18TH,AVE,None,3644_18TH_AVE


## 16.16 SMS Contact Activation Summary

This section creates an updated activation summary using the SimpleTexting contact export and classified replies.

This summary becomes the new checkpoint before phone-based customer matching.

In [15]:
sms_contact_activation_summary = pd.DataFrame([
    {"metric": "SimpleTexting Contacts", "value": len(contacts_address)},
    {"metric": "Contacts With Inbox Thread", "value": int(contacts_clean["has_inbox_thread"].sum())},
    {"metric": "Customer Replies", "value": int(contacts_replies["customer_replied_flag"].sum())},
    {"metric": "Verified Responses", "value": int(contacts_replies["verified_response_flag"].sum())},
    {"metric": "STOP Replies", "value": int((contacts_replies["reply_category"] == "STOP").sum())},
    {"metric": "Coupons Sent", "value": int(contacts_replies["coupon_sent_flag"].sum())},
    {"metric": "Follow-Up Needed", "value": int(contacts_replies["follow_up_needed"].sum())},
    {"metric": "Addresses Parsed", "value": int(contacts_address["address_join_key"].notna().sum())},
])

sms_contact_activation_summary.to_csv(
    CLEANED_SMS_DIR / "sms_contact_activation_summary.csv",
    index=False
)

sms_contact_activation_summary.to_csv(
    EXPORT_DIR / "sms_contact_activation_summary.csv",
    index=False
)

print("Exported:", CLEANED_SMS_DIR / "sms_contact_activation_summary.csv")
print("Exported:", EXPORT_DIR / "sms_contact_activation_summary.csv")

sms_contact_activation_summary

Exported: ../data/cleaned/sms/sms_contact_activation_summary.csv
Exported: ../data/exports/sms_contact_activation_summary.csv


,metric,value
0,SimpleTexting Contacts,498
1,Contacts With Inbox Thread,497
2,Customer Replies,65
3,Verified Responses,26
4,STOP Replies,39
5,Coupons Sent,26
6,Follow-Up Needed,0
7,Addresses Parsed,498


## 16.17 Next Step — Phone-Based Customer Match

The next section should match the SimpleTexting contact records back to the Pizza House customer dataset using `phone_clean`.

Planned next section:

```text
16.17 Phone-Based Customer Match
```

Planned logic:

- Identify the canonical Workbook 13 customer file
- Normalize customer phone numbers
- Join SimpleTexting contacts to customer records by `phone_clean`
- Calculate phone match rate
- Use `address_join_key` only as a secondary QA validation field

Phone becomes the canonical integration key for Workbook 16. Address becomes QA, not the primary match key.